In [1]:
# ===== Cell 1: 调用 Fess 文档搜索 API =====
# 踩坑说明：
#   Fess 15.6.1 里旧教程的两个地址都不能用了：
#     /json/           -> 已移除，返回 HTML"页面未找到"（所以 resp.json() 报 JSONDecodeError）
#     /api/v2/search   -> v2 接口是 Fess 15.7 以后才加入的，15.6.1 上同样 404
#   15.6.x 正确的 JSON 搜索接口是：
#     GET /api/v1/documents?q=关键词&num=10
import requests
import json

FESS_BASE = 'http://127.0.0.1:8080'
# 公开搜索一般不需要 token；如果给文档配置了权限/标签，才需要在后台生成 token
TOKEN = ''   # 例如 '220pMW3POp...'，没有权限控制就留空


def fess_search(keyword, num=10):
    '''通过 Fess API 搜索文档

    参数:
        keyword: 搜索关键词，如 '华为'
        num:     返回结果条数
    返回:
        (结果列表, 命中总数)；失败时返回 ([], 0) 并打印错误
    '''
    # 用 params 传参，requests 会自动对中文关键词做 URL 编码
    params = {'q': keyword, 'num': num, 'start': 0}
    if TOKEN:
        # 有权限控制时，v1 接口通过 URL 参数传 token
        params['token'] = TOKEN

    try:
        resp = requests.get(f'{FESS_BASE}/api/v1/documents',
                            params=params, timeout=10)
    except requests.RequestException as e:
        print(f'网络请求失败（Fess 启动了吗？）: {e}')
        return [], 0

    # 关键检查：先确认返回的真的是 JSON，而不是 HTML 错误页
    content_type = resp.headers.get('Content-Type', '')
    if resp.status_code != 200 or 'json' not in content_type:
        print(f'接口异常: HTTP {resp.status_code}, {content_type}')
        print('前200字:', resp.text[:200])
        return [], 0

    data = resp.json()
    results = data.get('data', [])          # 真正的搜索结果在 data 列表里
    total = data.get('record_count', 0)     # 命中总条数
    return results, total



In [2]:
# ===== Cell 2: 测试 Fess 搜索 =====
def show_results(keyword, num=10):
    results, total = fess_search(keyword, num)
    print(f'搜索「{keyword}」共命中 {total} 条，展示前 {len(results)} 条：\n')
    for i, item in enumerate(results, 1):
        # content_description 里可能带 <strong> 高亮标签，简单去掉方便阅读
        digest = item.get('content_description', '').replace('<strong>', '').replace('</strong>', '')
        print(f'{i}. {item.get("title")}')
        print(f'   路径: {item.get("site_path") or item.get("url")}')
        print(f'   相关度: {item.get("score")}')
        print(f'   摘要: {digest[:120]}...\n')


show_results('华为')



搜索「华为」共命中 1 条，展示前 1 条：

1. huawei1.txt
   路径: S:\PythonAILearningCourse\07-langchain-rag\txt\huawei1.txt
   相关度: 0.0090338215
   摘要:  华为数据存储产品线研发管理部部长常胜（右一）；智能存储与计算重点实验室沈志荣教授（右二）...probabilistic-logic barrier》技术成果。 在论坛现场，华为数据存储产品线研发管理部部长常胜同步公布了第七届“奥林帕斯奖...



In [3]:
# ===== Cell 3: 把 Fess 搜索封装成 Langchain 工具 =====
# 这样大模型 Agent 就能自己调用本地 Fess 文档库搜索资料
from langchain_core.tools import tool


@tool
def fess_doc_search(keyword: str) -> str:
    '''在本地 Fess 文档搜索服务器中检索文档。
    当用户需要查找本地资料库/文件中的内容、搜索文档时使用此工具。

    参数:
        keyword: 搜索关键词，例如 "华为"
    '''
    results, total = fess_search(keyword, num=5)
    if not results:
        return f'没有搜索到与「{keyword}」相关的文档。'
    lines = [f'共找到 {total} 条相关文档：']
    for i, item in enumerate(results, 1):
        digest = item.get('content_description', '').replace('<strong>', '').replace('</strong>', '')
        lines.append(f'{i}. 标题: {item.get("title")}\n   摘要: {digest[:150]}')
    return '\n'.join(lines)


# 直接调用验证工具
print(fess_doc_search.invoke({'keyword': '存储'}))



共找到 1 条相关文档：
1. 标题: huawei1.txt
   摘要:  华为数据存储产品线研发管理部部长常胜（右一）；智能存储与计算重点实验室沈志荣教授（右二） 作为数据存储领域的权威赛事...近日，在2026全球数据存储教授论坛上，第六届“奥林帕斯奖”获奖名单正式揭晓。来自国内外的7支顶尖科研团队凭借在数据存储领域的前沿研...
